# Notebook 11 — Multi-Scale Modeling

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 11 of 12  
**Time estimate:** 80 minutes

> Biology operates across scales: molecular reactions inside cells (microseconds),
> cell division and migration (hours), tissue organization (days), organ function
> (months). A multi-scale model couples these levels — agent behaviour depends on
> internal ODE state, and the population of agents feeds back on external conditions.

---
## Step 1 — Motivation

A tumor grows because individual cancer cells divide, migrate, and consume oxygen.
Their division rate depends on intracellular signaling (oncogene expression — an
ODE model). Their migration depends on oxygen gradients — which depend on how many
other cells are consuming oxygen nearby. No single-scale model captures all of this:
an ODE can't model cell movement; a PDE can't model individual cell decisions.
A hybrid ODE + ABM can.

---
## Step 2 — Intuition

**Hybrid ODE + ABM architecture:**
1. Each agent has an *internal state* governed by an ODE (e.g., metabolic activity,
   cell cycle protein levels).
2. The agent's *behaviour* (divide, die, move) depends on its internal state AND
   local external conditions (oxygen level, drug concentration).
3. External conditions are on a shared grid, updated as a PDE (diffusion +
   consumption by agents).

**Time scale separation:**
- Internal ODE: runs on a fine time step (e.g., Δt = 0.01 hours)
- Agent behaviour: updates on a coarser step (e.g., Δt = 0.5 hours)
- External field (PDE): runs on a medium step (e.g., Δt = 0.05 hours)
- The nested stepping scheme: for each outer ABM step, run the internal ODEs
  for each agent and update the diffusion field.

**Cellular Potts model (alternative):** energy-minimization-based cell shape
ABM — cells are connected patches of grid points, with energy terms for
adhesion, volume, and shape. CompuCell3D and Morpheus are specialized tools.
We use a simpler point-particle model here.

---
## Step 3 — Biological Background

**Tumor growth models:**
- ODE: Gompertzian growth N(t) = N_max exp(-exp(b - rt)). Fits data but no mechanism.
- PDE: reaction-diffusion of oxygen + nutrient, necrotic core formation.
- ABM: individual cell decisions → emergent tumor morphology (invasive fingers,
  necrotic core, proliferating rim).
- Hybrid: cells have internal ATP production ODEs + move along oxygen gradient +
  proliferate when ATP > threshold + die when oxygen < threshold.

**Oxygen in tumours:**
- Blood vessels deliver oxygen. Tumour cells consume it.
- Diffusion distance of oxygen in tissue: ~100–200 μm.
- Cells beyond this distance become hypoxic → activate HIF-1α → produce VEGF
  → signal for angiogenesis (new blood vessel growth).
- A multi-scale model can simulate: cells → HIF-1α activation (ODE) → VEGF
  secretion → new vessel formation (ABM decision) → oxygen restoration.

**Phoenix (PhysiCell) and BioFVM:**
Open-source platforms for multi-scale biological ABM with off-lattice cells
and PDE substrates (oxygen, drugs, cytokines). PhysiCell Studio provides a GUI.
These are the production-level tools; this notebook builds the concept from scratch.

---
## Step 4 — Mathematical Explanation

**Intracellular metabolic ODE (per cell):**
$$\frac{d[\text{ATP}]}{dt} = k_{\text{prod}} \cdot O_2(x, y) - k_{\text{consume}} \cdot [\text{ATP}]$$
Where $O_2(x,y)$ is the local oxygen concentration at the cell's position.
ATP drives proliferation: rate = $k_{\text{div}} \cdot \text{ATP} / (K_m + \text{ATP})$.

**Extracellular oxygen PDE (on grid):**
$$\frac{\partial O_2}{\partial t} = D_{O_2} \nabla^2 O_2 - \sum_{i} \alpha_i \delta(\mathbf{x} - \mathbf{x}_i)$$
where $\alpha_i$ is the oxygen consumption rate of cell $i$ at position $\mathbf{x}_i$.
On the grid, the consumption term becomes: for each occupied grid point,
subtract $\alpha \cdot n_{\text{cells}}$ from $O_2$.

**Cell division rule:**
$$P(\text{divide} | \text{ATP}) = k_{\text{div}} \cdot \frac{\text{ATP}}{K_m + \text{ATP}} \cdot \Delta t$$

**Cell death rule:**
$$P(\text{die} | O_2) = k_{\text{death}} \cdot \max(0, 1 - O_2 / O_2^{\text{thresh}}) \cdot \Delta t$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

rng = np.random.default_rng(42)

# ============================================================
# Hybrid ODE + ABM: Tumor growth model
# ============================================================

# ---- Constants ----
N_GRID   = 60       # grid size (each cell ≈ 25 μm)
N_STEPS  = 150      # simulation steps
DT       = 0.5      # hours per step

# Oxygen (O2) parameters
DO2      = 0.3      # diffusion coefficient (grid units²/step)
O2_supply = 1.0     # boundary condition (vessel O2 level)
O2_consume = 0.05   # per-cell consumption rate
O2_thresh = 0.15    # below this → hypoxic → apoptosis risk

# Cell metabolic parameters
ATP_prod    = 0.8   # ATP production rate (proportional to O2)
ATP_consume = 0.2   # baseline ATP consumption rate
Km_atp      = 0.5   # Michaelis-Menten constant
k_div       = 0.04  # division rate coefficient
k_death_hyp = 0.10  # death rate under hypoxia

# ---- Agent: Cancer Cell ----
class CancerCell:
    def __init__(self, row, col, atp=0.5):
        self.row = row
        self.col = col
        self.atp = atp
        self.alive = True

    def update_atp(self, o2_local, dt=DT):
        """Intracellular ATP dynamics ODE (Euler step)."""
        dATP = ATP_prod * o2_local - ATP_consume * self.atp
        self.atp = max(0, self.atp + dt * dATP)

    def division_probability(self, dt=DT):
        return k_div * self.atp / (Km_atp + self.atp) * dt

    def death_probability(self, o2_local, dt=DT):
        if o2_local < O2_thresh:
            return k_death_hyp * (1 - o2_local / O2_thresh) * dt
        return 0.0

# ---- Model ----
class TumorModel:
    def __init__(self, N=N_GRID, seed=42):
        self.N = N
        self.rng = np.random.default_rng(seed)
        self.cells = []
        self.O2 = np.ones((N, N)) * O2_supply  # initialize to full O2
        self.lap_kernel = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=float)
        self.step_count = 0
        self.history_n   = []  # cell count
        self.history_o2  = []  # mean O2
        self.snapshots   = []
        # Seed a small cluster of cells at the centre
        cx = cy = N // 2
        for dr in range(-2, 3):
            for dc in range(-2, 3):
                if abs(dr) + abs(dc) <= 2:
                    self.cells.append(CancerCell(cx+dr, cy+dc))

    def cell_grid(self):
        """Integer grid: number of cells per grid point."""
        grid = np.zeros((self.N, self.N), dtype=float)
        for c in self.cells:
            if c.alive:
                grid[c.row % self.N, c.col % self.N] += 1
        return grid

    def update_o2(self):
        """PDE: diffusion + consumption by cells. Dirichlet BC at boundary."""
        cg = self.cell_grid()
        lap = convolve2d(self.O2, self.lap_kernel, mode='same', boundary='fill', fillvalue=O2_supply)
        self.O2 = np.clip(self.O2 + DT * (DO2 * lap - O2_consume * cg), 0, O2_supply)
        # Dirichlet BC: fix O2 at boundary to O2_supply (vessels)
        self.O2[0, :] = self.O2[-1, :] = self.O2[:, 0] = self.O2[:, -1] = O2_supply

    def step(self):
        self.update_o2()
        new_cells = []
        dead = []
        # Shuffle agent order
        idxs = self.rng.permutation(len(self.cells))
        for i in idxs:
            cell = self.cells[i]
            if not cell.alive:
                dead.append(i)
                continue
            r, c = cell.row % self.N, cell.col % self.N
            o2_local = self.O2[r, c]
            # ATP update
            cell.update_atp(o2_local)
            # Death
            if self.rng.random() < cell.death_probability(o2_local):
                cell.alive = False
                dead.append(i)
                continue
            # Division
            if len(self.cells) < 800 and self.rng.random() < cell.division_probability():
                # Find a nearby empty grid point (Moore neighbourhood)
                offsets = [(dr, dc) for dr in [-1,0,1] for dc in [-1,0,1] if not (dr==0 and dc==0)]
                self.rng.shuffle(offsets)
                for dr, dc in offsets:
                    nr, nc = (r + dr) % self.N, (c + dc) % self.N
                    # Check if mostly empty
                    if self.O2[nr, nc] > O2_thresh * 0.5:
                        new_cells.append(CancerCell(nr, nc, atp=cell.atp * 0.5))
                        cell.atp *= 0.5  # split ATP
                        break
        self.cells = [c for c in self.cells if c.alive] + new_cells
        self.step_count += 1
        self.history_n.append(len(self.cells))
        self.history_o2.append(self.O2.mean())
        if self.step_count % 30 == 0:
            self.snapshots.append((self.step_count, self.cell_grid().copy(), self.O2.copy()))

print('Running tumor growth model (hybrid ODE + ABM)...')
model = TumorModel(N=N_GRID, seed=42)
for _ in range(N_STEPS):
    model.step()
print(f'Final cell count: {len(model.cells)}')
print(f'Mean O2 (final): {model.O2.mean():.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel A: Cell density snapshots (3 time points)
if len(model.snapshots) >= 3:
    s0 = model.snapshots[0]
    s1 = model.snapshots[len(model.snapshots)//2]
    s2 = model.snapshots[-1]
    combined = np.concatenate([s0[1], s1[1], s2[1]], axis=1)
    ax = axes[0, 0]
    ax.imshow(combined, cmap='hot', aspect='auto')
    ax.axvline(N_GRID, color='white', lw=1)
    ax.axvline(2*N_GRID, color='white', lw=1)
    ax.set_xticks([N_GRID//2, 3*N_GRID//2, 5*N_GRID//2])
    ax.set_xticklabels([f't={s0[0]}h', f't={s1[0]}h', f't={s2[0]}h'])
    ax.set_title('A. Tumor cell density (3 time points)\n(hot = many cells)')
else:
    axes[0,0].text(0.5, 0.5, 'Run more steps', ha='center', va='center',
                     transform=axes[0,0].transAxes)

# Panel B: Oxygen field (final)
ax = axes[0, 1]
im = ax.imshow(model.O2, cmap='Blues', origin='lower', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='O2 concentration')
cell_pos = np.array([(c.row, c.col) for c in model.cells if c.alive])
if len(cell_pos):
    ax.scatter(cell_pos[:, 1], cell_pos[:, 0], s=2, color='tomato', alpha=0.4, label='Cells')
ax.set_title('B. Oxygen field (final)\n(blue=high, red dots=cells)')
ax.legend(fontsize=8)

# Panel C: Cell count and O2 over time
ax = axes[0, 2]
ax.plot(range(len(model.history_n)), model.history_n, 'steelblue', lw=2, label='Cell count')
ax_r = ax.twinx()
ax_r.plot(range(len(model.history_o2)), model.history_o2, 'tomato', lw=2, label='Mean O2')
ax.set_xlabel('Step (× 0.5 hr)'); ax.set_ylabel('N cells', color='steelblue')
ax_r.set_ylabel('Mean O2', color='tomato')
ax.set_title('C. Tumor growth curve\nvs. mean oxygen level')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax_r.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=8)

# Panel D: ATP distribution of cells
ax = axes[1, 0]
atp_vals = [c.atp for c in model.cells if c.alive]
ax.hist(atp_vals, bins=30, color='green', alpha=0.7, edgecolor='black')
ax.axvline(Km_atp, color='tomato', ls='--', lw=1.5, label=f'Km_atp={Km_atp}')
ax.axvline(ATP_consume/ATP_prod, color='steelblue', ls=':', lw=1.5,
             label=f'O2=1 SS ATP={ATP_consume/ATP_prod:.2f}')
ax.set_xlabel('Intracellular ATP'); ax.set_ylabel('Cell count')
ax.set_title('D. ATP distribution (final)\n(cells near hypoxic core have low ATP)')
ax.legend(fontsize=8)

# Panel E: Multi-scale architecture diagram
ax = axes[1, 1]
ax.axis('off')
levels = [
    (0.85, 'INTRACELLULAR (ODE)\ndATP/dt = k_prod·O2 - k_consume·ATP\n→ division/death probability', 'lightgreen'),
    (0.55, 'CELL LEVEL (ABM)\nstep(): update ATP, divide, die, move\nscheduler: random activation', 'lightyellow'),
    (0.25, 'TISSUE LEVEL (PDE)\ndO2/dt = D∇²O2 - α·Σδ(x-xi)\ngrid updated each ABM step', 'lightblue'),
]
for y, text, color in levels:
    ax.text(0.5, y, text, ha='center', va='center', fontsize=9,
              bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.9),
              transform=ax.transAxes)
# Arrows
for y_top, y_bot in [(0.70, 0.65), (0.40, 0.35)]:
    ax.annotate('', xy=(0.5, y_bot), xytext=(0.5, y_top),
                   xycoords='axes fraction', textcoords='axes fraction',
                   arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.set_title('E. Hybrid multi-scale architecture')

# Panel F: Cell spatial distribution (scatter)
ax = axes[1, 2]
if len(cell_pos):
    atp_colors = np.array([c.atp for c in model.cells if c.alive])
    sc = ax.scatter(cell_pos[:, 1], cell_pos[:, 0],
                      c=atp_colors, cmap='RdYlGn', s=8, vmin=0, vmax=1)
    plt.colorbar(sc, ax=ax, label='Intracellular ATP')
ax.set_xlim(0, N_GRID); ax.set_ylim(0, N_GRID)
ax.set_title('F. Cell positions (colored by ATP)\n(low ATP = red = hypoxic core)')
ax.set_aspect('equal')

plt.suptitle('Module 15 NB11: Multi-Scale Modeling (Hybrid ODE + ABM)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('multiscale_modeling.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Add an apoptosis rule: cells with ATP below 0.1 for more than 10 consecutive
   steps trigger programmed death (not just probabilistic death). Compare the
   tumor growth curve with and without this rule.
2. Add a drug: at step 100, introduce a drug that reduces ATP_prod by 50% in all
   cells. Show the tumor growth curve before and after drug addition. Does the
   tumor shrink or just stop growing?
3. Add angiogenesis: if any cell's local O2 < O2_thresh * 0.5, it secretes VEGF.
   If VEGF concentration (diffusing on a grid) exceeds a threshold, add a new
   "vessel" source point that supplies O2 nearby. Show how this affects the
   O2 field and tumor growth.

---
## Step 10 — Quiz

1. What are the three components of a hybrid ODE + ABM model? Describe what
   each computes and how they are coupled.
2. Why is time-scale separation important in multi-scale modeling? What can go
   wrong if the intracellular ODE and the PDE run at very different time scales?
3. In the tumor model, why does a necrotic core form? Walk through the causal
   chain from oxygen diffusion to cell death to necrotic core formation.
4. What is PhysiCell and how does it extend the simple model in this notebook?
   Name two biological phenomena it has been used to study.

---
## Step 12 — Reflection

> *[In one paragraph: explain what a multi-scale model reveals about tumor biology
> that a single-scale ODE (e.g., Gompertzian growth) cannot. Give two specific
> biological phenomena that emerge from the hybrid model — phenomena that are
> invisible in the ODE and require individual cell-level simulation.]*

---
*Next: `12_mini_project_and_assessment.ipynb`*